# CyberAccountant pipeline — interactive driver

Run cells top to bottom. Each section runs one stage and shows its artifact.

Config: change `ID_STYLE` between `"mnemonic"` and `"numeric"` to compare.

In [ ]:
from pathlib import Path
from cyberaccountant import Pipeline, PipelineConfig

ID_STYLE = "mnemonic"          # or "numeric"
TAG_MODEL = "deepseek/deepseek-v4.1-flash-20261017"
# NB: the old "~deepseek/deepseek-pro-latest" id 404s on this account —
# relations run fine on the flash model too.
REL_MODEL = TAG_MODEL

cfg = PipelineConfig(
    data_dir=Path("../data_md"),
    out_dir=Path("../out"),
    id_style=ID_STYLE,
    tag_model=TAG_MODEL,
    rel_model=REL_MODEL,
)
p = Pipeline(cfg)

## Stage 0 — ingest / rename / dedup

In [2]:
mapping = p.ingest()
import pandas as pd
pd.DataFrame(mapping).T

,original,role,sha256,duplicate_of,tag
478f70c0,данные/02_01_Ведомость_ЗП_Январь_2026.xlsx.md,data,4aa8154585ee1c226d3434b25fed004d2463a6e5e5c195...,NaN,NaN
f7173045,данные/02_02_Ведомость_ЗП_Февраль_2026.xlsx.md,data,c4ccaff6c874f63c26af97d4de448f9a0dc3ac1f811349...,NaN,NaN
feb8e6ab,данные/02_03_Ведомость_ЗП_Март_2026.xlsx.md,data,5e25554b83e1c434ef4c6befdc0b43ebba05f8ce83bdd9...,NaN,NaN
6fd3d91d,данные/02_04_Ведомость_ЗП_Апрель_2026.xlsx.md,data,4e108b7cc42f3d987d779e1c255bc3403ed969cde1c52e...,NaN,NaN
f37ebfc3,данные/02_05_Ведомость_ЗП_Май_2026.xlsx.md,data,823f89a747de7db43d6a8c25ee98cb00b92bea18879527...,NaN,NaN
ba317b65,данные/03_01_Страховые_взносы_Январь_2026.xlsx.md,data,f577cc50ec3cebb0f47b7a5f14f511871dd4e3723af3a8...,NaN,NaN
312598a2,данные/03_02_Страховые_взносы_Февраль_2026.xls...,data,77d1da207f88b9eef815aabfb42b2e7c6f5362403d1007...,NaN,NaN
287e47c4,данные/03_03_Страховые_взносы_Март_2026.xlsx.md,data,cb481eef1a047a8cecf7d255b900efbe9c73a4bf48181a...,NaN,NaN
4bfc9ad2,данные/03_04_Страховые_взносы_Апрель_2026.xlsx.md,data,b988a7139882f844ed169c3f7dc0c5c64896f56bd30411...,NaN,NaN
83b5cc17,данные/03_05_Страховые_взносы_Май_2026.xlsx.md,data,8414801ec16918fd770ae8505a267e967480b80c13d94c...,NaN,NaN


## Stage 1 — entity tagging (LLM, per file)

In [3]:
from cyberaccountant.stage1_tag import TagProgress, TagStats


def on_tag_progress(prog: TagProgress, stats: TagStats) -> None:
    """Swappable live-progress hook (forward to a frontend here later)."""
    if prog.ok:
        line = (
            f"[{stats.done}/{stats.total}] OK  {prog.file}  "
            f"entities={prog.entities} retries={prog.retries}"
        )
    else:
        line = f"[{stats.done}/{stats.total}] ERR {prog.file}  {prog.error}"
    print(line, flush=True)


tagged = p.tag(on_progress=on_tag_progress)
print(f"\nDone: {len(tagged)} files tagged.")
list(tagged.keys())[:5], len(tagged)

[1/24] OK  ba317b65  entities=6 retries=0
[2/24] OK  478f70c0  entities=9 retries=0
[3/24] OK  feb8e6ab  entities=9 retries=0
[4/24] OK  4bfc9ad2  entities=6 retries=0
[5/24] OK  f37ebfc3  entities=8 retries=1
[6/24] OK  f7173045  entities=9 retries=1
[7/24] OK  6fd3d91d  entities=8 retries=1
[8/24] OK  ca042061  entities=6 retries=0
[9/24] ERR 312598a2  tagging failed for 312598a2: rebuilt doc diverges at char 146: expected '\\| Период: Февраль 2026 |  |  |  |\n|  | ', got '| Период: Февраль 2026 |  |  |  |\n|  |  '
[10/24] OK  102633fa  entities=6 retries=0
[11/24] OK  44ec4d45  entities=5 retries=0
[12/24] OK  b8394e39  entities=5 retries=0
[13/24] OK  a3441096  entities=4 retries=0
[14/24] OK  da9c6253  entities=4 retries=0
[15/24] OK  95f0da3e  entities=4 retries=0
[16/24] OK  60f3cf82  entities=5 retries=0
[17/24] OK  a23da771  entities=5 retries=0
[18/24] OK  e69a555c  entities=4 retries=0
[19/24] OK  83b5cc17  entities=6 retries=2
[20/24] OK  0cece54d  entities=4 retries=0
[21/

(['ba317b65', '478f70c0', 'feb8e6ab', '4bfc9ad2', 'f37ebfc3'], 23)

## Stage 2 — global entity registry

In [4]:
reg = p.registry()
import pandas as pd
rows = [{"num": e.num, "mnemonic": e.mnemonic, "type": e.type,
         "raw": e.raw, "value": e.value, "unit": e.unit, "file": e.file,
         "role": e.role} for e in reg.entities]
df = pd.DataFrame(rows)
df.head(20)

/home/alice/code/shitposting/CyberAccountant/src/cyberaccountant/bip39.py:90: RuntimeWarning: BIP39 wordlist not found; using deterministic generated fallback. For the canonical list run: python scripts/fetch_bip39.py
  self._a, self._b = load_words()


,num,mnemonic,type,raw,value,unit,file,role
0,0,dere-gidy,docid,03-26,03-26,,0cece54d,data
1,1,cusa-zypu,date,март 2026,None,,0cece54d,data
2,2,vybi-jiku,contract,24/03/15-01,24/03/15-01,,0cece54d,data
3,3,nyce-jamo,amount,"703 244,51",703244.51,RUB,0cece54d,data
4,4,hela-gaqu,docid,КЛ-31,КЛ-31,,102633fa,data
5,5,vexa-refo,date,10.04.2026,739716,,102633fa,data
6,6,mala-siry,contract,24/03/15-01,24/03/15-01,,102633fa,data
7,7,qeba-faqu,number,3,3,,102633fa,data
8,8,rade-gydo,docid,КЛ-31,КЛ-31,,102633fa,data
9,9,fubi-basy,amount,"270 000,00",270000.00,RUB,102633fa,data


In [5]:
df["type"].value_counts()

type
amount      83
number      57
date        32
contract    24
docid       22
Name: count, dtype: int64

## Stage 2b — date normalization (LLM, one batch call)

The deterministic parser handles `DD.MM.YYYY` and `DD month YYYY`, but not
period labels like `Январь 2026` — those entities get `value = None` and
would stay unknowns in the solver. This stage asks the model for concrete
dates (`Январь 2026` → `01.01.2026`, first day of the period). Provenance
is kept in `Entity.normalized_from`.

In [ ]:
changed = p.normalize_dates()
print(f"normalized {len(changed)} date entities")
[(e.mnemonic, e.normalized_from, e.value) for e in changed]

## Stage 3 — replace tags with ids

In [6]:
replaced = p.replace()
h8 = next(iter(replaced))
print(replaced[h8][:1200])

Справка о расходах ИТ <e id="dere-gidy">03-26</e>

Период: <e id="cusa-zypu">март 2026</e>

Договор: <e id="vybi-jiku">24/03/15-01</e>

Подразделение: административный контур проекта

Настоящей справкой подтверждаются расходы на обеспечение ИТ-инфраструктуры, программных лицензий и сервисов совместной работы, используемых административной группой договора.

Расходы не относятся непосредственно к отдельной производственной операции и учитываются в составе общехозяйственного обеспечения.

| Описание | Сумма, руб. |
| --- | --- |
| Инфраструктура, лицензии и сервисы | <e id="nyce-jamo">703 244,51</e> |




## Stage 4 — scope checks (contract / period)

In [32]:
scope = p.scope()
print("target label: ", scope.get("target_label"))
print("target row entities:", len(scope.get("target_row") or []),
      "(rest of the card becomes trusted constants)")
print("oos:", {k: v["reasons"] for k, v in scope["oos"].items()})
scope.get("target_row")

target label:  None
target row entities: 0 (rest of the card becomes trusted constants)
oos: {'60f3cf82': ['period'], 'a23da771': ['contract']}


## Stage 5 — relation extraction (LLM, whole corpus)

In [33]:
# rel_model is already set to the tag model in the config cell above
# (the old "~deepseek/deepseek-pro-latest" id 404s on this account).
# Switch here if you want to A/B another model for relation extraction:
# p.cfg.rel_model = "deepseek/deepseek-v4.1-flash-20261017"

In [34]:
relations_raw = p.relations()
len(relations_raw), relations_raw[:5]

(75,
 [{'equation': 'vybi-jiku == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'},
  {'equation': 'mala-siry == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'},
  {'equation': '01.01.2026 <= vexa-refo && vexa-refo <= 31.05.2026',
   'tag': 'period',
   'source': 'scope'},
  {'equation': 'qyqu-niki == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'},
  {'equation': 'jaxa-jyve == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'}])

## Stage 6 — solve the system

In [35]:
rels, fishy, verdicts, dangling = p.solve()   # scope -> only target row stays variable
import pandas as pd
rows = [{"i": r.index, "equation": r.equation, "tag": r.tag,
         "status": r.status, "touches_card": r.touches_card,
         "error": r.error} for r in rels]
df = pd.DataFrame(rows)
print(df["status"].value_counts().to_dict(), "| card-touching:",
      df[df.touches_card]["status"].value_counts().to_dict())
df[df.status != "true"]

TypeError: solve_system() got an unexpected keyword argument 'variable_keys'

## Stage 7 — report

In [ ]:
report = p.report()
from cyberaccountant.report import format_report_text
print(format_report_text(report, p._mapping))

In [25]:
import pandas as pd
pd.DataFrame(report["card"]["cells"])

,id,num,mnemonic,raw,type,status,stated,expected,diff,file
0,gejo-toto,72,gejo-toto,3/1,docid,unverified,3/1,NaN,NaN,9ce4b2a0
1,ceme-weka,73,ceme-weka,Август 2025,date,unverified,NaN,NaN,NaN,9ce4b2a0
2,foge-kihi,74,foge-kihi,Май 2026,date,unverified,NaN,NaN,NaN,9ce4b2a0
3,bagu-tudi,75,bagu-tudi,августа 2025,date,unverified,NaN,NaN,NaN,9ce4b2a0
4,popo-nymi,76,popo-nymi,мая 2026,date,unverified,NaN,NaN,NaN,9ce4b2a0
...,...,...,...,...,...,...,...,...,...,...
83,gydi-citi,155,gydi-citi,"72 782,15",amount,unverified,72782.15,NaN,NaN,9ce4b2a0
84,wire-sijy,156,wire-sijy,"93 237,27",amount,unverified,93237.27,NaN,NaN,9ce4b2a0
85,zuxu-reqi,157,zuxu-reqi,"1 198 231,97",amount,unverified,1198231.97,NaN,NaN,9ce4b2a0
86,cavu-foja,158,cavu-foja,"3 276 676,77",amount,unverified,3276676.77,NaN,NaN,9ce4b2a0


In [ ]:
# raw JSON summaries if you need them programmatically
import json
print(json.dumps(report["summary"], indent=2, ensure_ascii=False))